# World Cup match data cleaning

This notebook prepares a dataset for 3-class match prediction. It uses only information available before kickoff to avoid leakage. The target is 0 for a Team A win, 1 for a draw, and 2 for a Team B win.

This notebook independently loads the raw World Cup dataset and prepares the final pre-match dataset for machine learning. EDA is performed separately in `1_EDA.ipynb` and is used to understand the data and identify potential issues, but its temporary analysis columns are not used as inputs to this notebook.

## 1. Imports and paths

These libraries handle files, text, tables, and the target variable.

In [1]:
from pathlib import Path
import shutil
import unicodedata
import pandas as pd
import numpy as np

raw_dir = Path('../data/raw')
processed_dir = Path('../data/processed')
processed_dir.mkdir(parents=True, exist_ok=True)
print('Paths ready.')

Paths ready.


## 2. Copy the raw matches file to processed

The raw file is never changed. Each run starts from a fresh copy in the processed folder, and all cleaning steps work on that copy.

## Missing-value decisions

Most missing values are in match-event columns, xG, penalties, cards, officials, and notes. These describe the current match or have very incomplete historical coverage, so they are removed rather than filled. The retained pre-match context columns are checked below; if a retained categorical value is missing, it is labeled `Unknown`. Missing rolling history means that a team has no previous matches, so those numeric features are set to zero, which represents no recorded prior history.

In [2]:
source_path = raw_dir / 'matches_1930_2022.csv'
working_path = processed_dir / 'matches_1930_2022.csv'
shutil.copy2(source_path, working_path)
print(f'Copied raw matches to: {working_path}')

Copied raw matches to: ..\data\processed\matches_1930_2022.csv


## 3. Load the raw matches dataset

The raw file must contain all original columns because scores and match events are needed to create historical features.

In [3]:
matches_df = pd.read_csv(working_path)
print(f'Raw matches shape: {matches_df.shape}')
if matches_df.shape[1] < 44:
    raise ValueError('The raw matches file must contain at least 44 original columns.')
print('Raw dataset loaded.')
# Playoff-team historical results used as post-tournament strength data.
rating_path = raw_dir / 'Rating_wc.csv'
rating_df = pd.read_csv(rating_path)
rating_df.columns = ['Year', 'Team', 'Final_Position', 'Eliminated_Round', 'Rating']
print(f'Raw playoff results shape: {rating_df.shape}')


Raw matches shape: (964, 44)
Raw dataset loaded.
Raw playoff results shape: (307, 5)


## 4. Clean text encoding

Text is normalized so names with special characters use a consistent format.

In [4]:
def clean_text(value):
    if not isinstance(value, str):
        return value
    value = unicodedata.normalize('NFKC', value)
    value = value.replace('\u00c2', '')
    value = value.replace('\u00a0', ' ')
    value = value.replace('\ufeff', '')
    return ' '.join(value.split()).strip()

text_columns = matches_df.select_dtypes(include='object').columns
for column_name in text_columns:
    matches_df[column_name] = matches_df[column_name].map(clean_text)
print('Text encoding cleaned.')

Text encoding cleaned.


## 5. Normalize team names

Historical names are mapped to modern names so the same team is treated consistently.

In [5]:
matches_df = matches_df.rename(columns={'home_team': 'team_A', 'away_team': 'team_B'})
team_name_map = {
    'West Germany': 'Germany',
    'Soviet Union': 'Russia',
    'IR Iran': 'Iran',
    'Korea Republic': 'South Korea',
    'Korea DPR': 'North Korea',
    'China PR': 'China',
    "CÃ´te d'Ivoire": 'Ivory Coast',
    'TÃ¼rkiye': 'Turkey',
    'Republic of Ireland': 'Ireland',
    'Bosnia and Herzegovina': 'Bosnia'
}
unique_before = set()
for column_name in ['team_A', 'team_B']:
    unique_before.update(matches_df[column_name].dropna().unique())
for column_name in ['team_A', 'team_B', 'Host']:
    if column_name in matches_df.columns:
        matches_df[column_name] = matches_df[column_name].replace(team_name_map)
unique_after = set()
for column_name in ['team_A', 'team_B']:
    unique_after.update(matches_df[column_name].dropna().unique())
print(f'Unique teams before: {len(unique_before)}')
print(f'Unique teams after: {len(unique_after)}')
print('Team names normalized.')
# Clean and normalize the playoff-results table.
rating_text_columns = rating_df.select_dtypes(include='object').columns
for column_name in rating_text_columns:
    rating_df[column_name] = rating_df[column_name].map(clean_text)
rating_df['Year'] = pd.to_numeric(rating_df['Year'], errors='coerce')
rating_df['Final_Position'] = pd.to_numeric(rating_df['Final_Position'], errors='coerce')
rating_df['Rating'] = pd.to_numeric(rating_df['Rating'], errors='coerce')
rating_df = rating_df.dropna(subset=['Year', 'Team', 'Final_Position']).copy()
rating_df['Year'] = rating_df['Year'].astype(int)
rating_df['Final_Position'] = rating_df['Final_Position'].astype(int)
rating_df['Team'] = rating_df['Team'].replace(team_name_map)
rating_df = rating_df.drop_duplicates(['Year', 'Team'])
print(f'Cleaned playoff results shape: {rating_df.shape}')


Unique teams before: 86
Unique teams after: 84
Team names normalized.
Cleaned playoff results shape: (266, 5)


## 6. Create the target variable

The target is created before scores are removed. Scores are not allowed to remain as model features because they describe the match outcome.

In [6]:
# Add the latest completed World Cup result for each team as a pre-match feature.
# We deliberately use only tournaments before the current match year to avoid
# leaking the current tournament's final position into its own predictions.
rating_records = rating_df.sort_values(['Year', 'Team']).to_dict('records')
rating_history = {}
for rating_row in rating_records:
    rating_history.setdefault(rating_row['Team'], []).append(rating_row)

def prior_world_cup_rating(team, year):
    prior = [item for item in rating_history.get(team, []) if item['Year'] < year]
    if not prior:
        return (0, 0, 'Unknown', 0.0)
    latest = prior[-1]
    # Higher values mean stronger historical tournament performance.
    strength = latest['Rating']
    return (latest['Final_Position'], 1, latest['Eliminated_Round'], strength)

rating_features = []
for _, match_row in matches_df.iterrows():
    a = prior_world_cup_rating(match_row['team_A'], int(match_row['Year']))
    b = prior_world_cup_rating(match_row['team_B'], int(match_row['Year']))
    rating_features.append({
        'A_final_position': a[0], 'B_final_position': b[0],
        'A_rating_available': a[1], 'B_rating_available': b[1],
        'A_eliminated_round': a[2], 'B_eliminated_round': b[2],
        'A_post_wc_rating': a[3], 'B_post_wc_rating': b[3],
        'post_wc_rating_diff': a[3] - b[3]
    })
matches_df = pd.concat([matches_df.reset_index(drop=True), pd.DataFrame(rating_features)], axis=1)
print('Historical Rating_wc features added without current-tournament leakage.')

# 0 = Team A Win, 1 = Draw, 2 = Team B Win
matches_df['target'] = np.where(
    matches_df['home_score'] > matches_df['away_score'],
    0,
    np.where(matches_df['home_score'] == matches_df['away_score'], 1, 2)
)
print('Target counts:')
print(matches_df['target'].value_counts().sort_index())
print('Target created.')

Historical Rating_wc features added without current-tournament leakage.
Target counts:
target
0    532
1    214
2    218
Name: count, dtype: int64
Target created.


## 7. Parse dates and sort chronologically

Chronological order is required so each match can use only earlier history.

In [7]:
matches_df['date'] = pd.to_datetime(matches_df['Date'], errors='coerce')
missing_dates = matches_df['date'].isna().sum()
print(f'Missing dates: {missing_dates}')
matches_df = matches_df.dropna(subset=['date'])
matches_df = matches_df.sort_values('date').reset_index(drop=True)
print(f'Sorted matches shape: {matches_df.shape}')
print('Date preparation done.')

Missing dates: 0
Sorted matches shape: (964, 55)
Date preparation done.


## 8. Build rolling features without time leakage

For every match, only records with a date strictly earlier than the current date are used. This prevents future information from entering the features. The features describe the last five matches and the total number of earlier matches for each team. Host flags describe the pre-match context.

In [8]:
matches_df['home_score'] = pd.to_numeric(matches_df['home_score'], errors='coerce')
matches_df['away_score'] = pd.to_numeric(matches_df['away_score'], errors='coerce')
history = {}
feature_rows = []

for row_index, match_row in matches_df.iterrows():
    feature_row = {}
    team_sides = [('A', match_row['team_A']), ('B', match_row['team_B'])]
    for team_side, team_name in team_sides:
        past_matches = []
        for past_match in history.get(team_name, []):
            if past_match['date'] < match_row['date']:
                past_matches.append(past_match)
        last_five = past_matches[-5:]
        goals_scored = sum(item['scored'] for item in last_five)
        goals_conceded = sum(item['conceded'] for item in last_five)
        wins = sum(item['result'] == 1 for item in last_five)
        if last_five:
            win_rate = wins / len(last_five)
        else:
            win_rate = 0
        feature_row[f'{team_side}_goals_scored_last_5'] = goals_scored
        feature_row[f'{team_side}_goals_conceded_last_5'] = goals_conceded
        feature_row[f'{team_side}_win_rate_last_5'] = win_rate
        feature_row[f'{team_side}_matches_played_total'] = len(past_matches)
    feature_row['is_host_A'] = int(match_row['team_A'] == match_row['Host'])
    feature_row['is_host_B'] = int(match_row['team_B'] == match_row['Host'])
    feature_rows.append(feature_row)
    home_score = match_row['home_score']
    away_score = match_row['away_score']
    if home_score > away_score:
        result_A, result_B = 1, -1
    elif home_score == away_score:
        result_A, result_B = 0, 0
    else:
        result_A, result_B = -1, 1
    home_history = {'date': match_row['date'], 'scored': home_score, 'conceded': away_score, 'result': result_A}
    away_history = {'date': match_row['date'], 'scored': away_score, 'conceded': home_score, 'result': result_B}
    if match_row['team_A'] not in history:
        history[match_row['team_A']] = []
    if match_row['team_B'] not in history:
        history[match_row['team_B']] = []
    history[match_row['team_A']].append(home_history)
    history[match_row['team_B']].append(away_history)

feature_df = pd.DataFrame(feature_rows)
matches_df = pd.concat([matches_df, feature_df], axis=1)
print(f'Rolling features added. New shape: {matches_df.shape}')

Rolling features added. New shape: (964, 65)


## 9. Drop leakage columns

xG, penalties, cards, own goals, scores, and event columns describe the current match. Attendance is not known before kickoff. These columns are removed after the rolling features have been calculated.

In [9]:
leaky_columns = [
    'home_xg', 'away_xg', 'home_penalty', 'away_penalty',
    'home_penalty_goal', 'away_penalty_goal',
    'home_penalty_shootout_goal_long', 'away_penalty_shootout_goal_long',
    'home_red_card', 'away_red_card', 'home_yellow_red_card', 'away_yellow_red_card',
    'home_own_goal', 'away_own_goal', 'Attendance', 'home_score', 'away_score', 'Score',
    'home_goal', 'away_goal', 'home_goal_long', 'away_goal_long',
    'home_yellow_card_long', 'away_yellow_card_long',
    'home_substitute_in_long', 'away_substitute_in_long',
    'Notes', 'Referee', 'Officials', 'home_captain', 'away_captain'
]
existing_leaky_columns = []
for column_name in leaky_columns:
    if column_name in matches_df.columns:
        existing_leaky_columns.append(column_name)
matches_df = matches_df.drop(columns=existing_leaky_columns)
print(f'Dropped {len(existing_leaky_columns)} leakage columns.')
print(f'Shape after leakage removal: {matches_df.shape}')

Dropped 31 leakage columns.
Shape after leakage removal: (964, 34)


## 10. Select final pre-match columns

The final dataset keeps context known before kickoff, rolling history features, and the target.

In [10]:
final_columns = [
    'team_A', 'team_B', 'Date', 'Year', 'Host', 'Round', 'Venue',
    'home_manager', 'away_manager', 'target',
    'A_goals_scored_last_5', 'A_goals_conceded_last_5', 'A_win_rate_last_5', 'A_matches_played_total',
    'B_goals_scored_last_5', 'B_goals_conceded_last_5', 'B_win_rate_last_5', 'B_matches_played_total',
    'is_host_A', 'is_host_B', 'A_final_position', 'B_final_position', 'A_eliminated_round', 'B_eliminated_round', 'A_rating_available', 'B_rating_available', 'A_post_wc_rating', 'B_post_wc_rating', 'post_wc_rating_diff'
]
available_columns = []
for column_name in final_columns:
    if column_name in matches_df.columns:
        available_columns.append(column_name)
final_df = matches_df[available_columns].copy()
final_df = final_df.dropna(subset=['team_A', 'team_B', 'Date', 'target'])
print(f'Final shape: {final_df.shape}')
print(f'Final columns: {final_df.columns.tolist()}')

Final shape: (964, 29)
Final columns: ['team_A', 'team_B', 'Date', 'Year', 'Host', 'Round', 'Venue', 'home_manager', 'away_manager', 'target', 'A_goals_scored_last_5', 'A_goals_conceded_last_5', 'A_win_rate_last_5', 'A_matches_played_total', 'B_goals_scored_last_5', 'B_goals_conceded_last_5', 'B_win_rate_last_5', 'B_matches_played_total', 'is_host_A', 'is_host_B', 'A_final_position', 'B_final_position', 'A_eliminated_round', 'B_eliminated_round', 'A_rating_available', 'B_rating_available', 'A_post_wc_rating', 'B_post_wc_rating', 'post_wc_rating_diff']


In [11]:
# Columns used by the modeling stage. Date is kept for chronological splitting, not as a feature.
feature_columns = [
    'team_A', 'team_B', 'Year', 'Host', 'Round', 'Venue',
    'home_manager', 'away_manager',
    'A_goals_scored_last_5', 'A_goals_conceded_last_5', 'A_win_rate_last_5', 'A_matches_played_total',
    'B_goals_scored_last_5', 'B_goals_conceded_last_5', 'B_win_rate_last_5', 'B_matches_played_total',
    'is_host_A', 'is_host_B'
]
feature_columns += ['A_final_position', 'B_final_position', 'A_rating_available', 'B_rating_available', 'A_post_wc_rating', 'B_post_wc_rating', 'post_wc_rating_diff']
missing_feature_columns = [col for col in feature_columns if col not in final_df.columns]
if missing_feature_columns:
    raise ValueError(f'Missing required feature columns: {missing_feature_columns}')

categorical_features = ['team_A', 'team_B', 'Host', 'Round', 'Venue', 'home_manager', 'away_manager']
for column_name in categorical_features:
    if column_name in final_df.columns:
        final_df[column_name] = final_df[column_name].fillna('Unknown')

history_features = [col for col in feature_columns if 'last_5' in col or 'matches_played' in col]
final_df[['A_final_position','B_final_position','A_rating_available','B_rating_available','A_post_wc_rating','B_post_wc_rating','post_wc_rating_diff']] = final_df[['A_final_position','B_final_position','A_rating_available','B_rating_available','A_post_wc_rating','B_post_wc_rating','post_wc_rating_diff']].fillna(0)
final_df[history_features] = final_df[history_features].fillna(0)

unexpected_missing = final_df[feature_columns].isna().sum()
unexpected_missing = unexpected_missing[unexpected_missing > 0]
if not unexpected_missing.empty:
    raise ValueError(f'Unexpected missing values in features: {unexpected_missing.to_dict()}')
print('All modeling features exist and contain no unexpected missing values.')

All modeling features exist and contain no unexpected missing values.


## 11. Validate the final dataset

These checks confirm that the target is valid, required identifiers are present, and no missing values are hidden.

In [12]:
assert set(final_df['target'].unique()).issubset({0, 1, 2})
assert final_df[['team_A', 'team_B', 'Date', 'target']].notna().all().all()
remaining_missing = final_df.isna().sum()
print('Target counts:')
print(final_df['target'].value_counts().sort_index())
print('Columns with missing values:')
print(remaining_missing[remaining_missing > 0])
print('Validation passed.')

Target counts:
target
0    532
1    214
2    218
Name: count, dtype: int64
Columns with missing values:
Series([], dtype: int64)
Validation passed.


## 12. Save the cleaned dataset

This is the single dataset used by the modeling notebook. No intermediate CSV files are saved.

In [13]:
output_path = processed_dir / 'matches_1930_2022_cleaned.csv'
final_df.to_csv(output_path, index=False)
print(f'Saved cleaned dataset: {output_path}')
print(f'Saved shape: {final_df.shape}')
print(f'Saved columns: {final_df.columns.tolist()}')

Saved cleaned dataset: ..\data\processed\matches_1930_2022_cleaned.csv
Saved shape: (964, 29)
Saved columns: ['team_A', 'team_B', 'Date', 'Year', 'Host', 'Round', 'Venue', 'home_manager', 'away_manager', 'target', 'A_goals_scored_last_5', 'A_goals_conceded_last_5', 'A_win_rate_last_5', 'A_matches_played_total', 'B_goals_scored_last_5', 'B_goals_conceded_last_5', 'B_win_rate_last_5', 'B_matches_played_total', 'is_host_A', 'is_host_B', 'A_final_position', 'B_final_position', 'A_eliminated_round', 'B_eliminated_round', 'A_rating_available', 'B_rating_available', 'A_post_wc_rating', 'B_post_wc_rating', 'post_wc_rating_diff']


## 13. Time-aware evaluation plan

Train on matches with `Year <= 2018`. Validate on matches from `Year == 2022`. For World Cup 2026, predict one match at a time and add each completed match to history before predicting the next match. Do not use a random split because it can expose future information to training.

In [ ]:
# Final cleaned dataset preview for inspection and screenshot.
print(f'Final cleaned dataset shape: {final_df.shape}')
display(final_df.head())
display(final_df.isna().sum().rename('missing_values').to_frame())